## **Mount Google Drive & Environment Setup**

In [1]:
from google.colab import drive

# 1. Mount Google Drive to 'Papers' folder
drive.mount('/content/drive')

# 2. Install required Python packages for PDF text extraction and NLP analysis
!pip install pymupdf tqdm pandas nltk spacy textstat
!python -m spacy download en_core_web_sm

# 3. Download NLTK tokenisation models silently
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("[✓] Cell 1 Complete: Google Drive connected & dependencies ready!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 56.7 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 112.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 61.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pymupdf]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 31.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
[✓] Cell 1 Complete: Google Drive connected & dependencies ready!


## **PDF Text Extractor & Cleaner**
* Scans Papers folder for PDF files, extracts the raw text block-by-block using PyMuPDF, cleans up broken line-hyphenations (e.g. analys-\nis $\rightarrow$ analysis) and page headers/footers, and saves the cleaned text files to Processed_Text.

In [2]:
import os
import re
from pathlib import Path
import fitz  # PyMuPDF library for fast PDF text reading
from tqdm import tqdm  # Visual progress bar

class PaperTextProcessor:
    """Extracts raw text from research PDFs and cleans formatting noise."""
    def __init__(self, papers_dir: str, output_txt_dir: str):
        self.papers_dir = Path(papers_dir)
        self.output_txt_dir = Path(output_txt_dir)
        self.output_txt_dir.mkdir(parents=True, exist_ok=True)

    def extract_and_clean_all(self) -> list:
        """Locates all PDF files in the target folder and converts them."""
        pdf_files = list(self.papers_dir.glob("**/*.pdf"))
        print(f"\n[+] Found {len(pdf_files)} PDF paper(s) in: {self.papers_dir}")
        processed_records = []
        for pdf_path in tqdm(pdf_files, desc="Extracting & Cleaning PDFs"):
            clean_text = self._convert_and_clean_single_pdf(pdf_path)
            if clean_text:
                # Save individual cleaned .txt file preserving subfolder structure
                relative_path = pdf_path.relative_to(self.papers_dir)
                txt_path = self.output_txt_dir / relative_path.with_suffix(".txt")
                txt_path.parent.mkdir(parents=True, exist_ok=True)
                with open(txt_path, "w", encoding="utf-8") as f:
                    f.write(clean_text)
                processed_records.append({
                    "filename": pdf_path.name,
                    "txt_path": txt_path,
                    "clean_text": clean_text
                })
        return processed_records

    def _convert_and_clean_single_pdf(self, pdf_path: Path) -> str:
        """Opens PDF file and extracts text page-by-page."""
        try:
            doc = fitz.open(str(pdf_path))
            raw_text_blocks = [page.get_text("text") for page in doc if page.get_text("text")]
            doc.close()
            return self._clean_text("\n".join(raw_text_blocks))
        except Exception as e:
            print(f"[-] Error converting {pdf_path.name}: {e}")
            return ""

    def _clean_text(self, text: str) -> str:
        """Removes PDF artifact noise: line-break hyphens, page numbers, and extra spaces."""
        # 1. Rejoin words split by hyphen at end of line
        text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
        # 2. Remove isolated page numbers surrounded by newlines
        text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
        # 3. Collapse multiple spaces and line breaks into standard spacing
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n+', '\n', text)
        return text.strip()

print("[✓] Cell 2 Complete: PaperTextProcessor defined!")

[✓] Cell 2 Complete: PaperTextProcessor defined!


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


### **Stylometric Feature Extractor Engine**

* Analyses each paper's text across 4 core linguistic dimensions:
  1. **Structure &amp; Vocabulary Diversity**: Word count, sentence count, Type-Token Ratio (TTR), average word/sentence length.
  2. **Grammar &amp; Part-of-Speech Ratios**: Proportion of Nouns, Verbs, and Adjectives via `spaCy`.
  3. **Readability Scores**: Flesch Reading Ease, Gunning Fog Index, and SMOG Index via `textstat`.
  4. **Information Density (Entropy)**: Character and Word Shannon Entropy for predictability.

In [3]:
import math
import string
from collections import Counter
import pandas as pd
import numpy as np
import nltk
import spacy
import textstat

# Load spaCy small English pipeline for Part-of-Speech (POS) tagging
nlp = spacy.load("en_core_web_sm", disable=["ner", "textcat"])

class StylometricAnalyzer:
    """Calculates quantitative writing style, readability, and information entropy."""
    def analyze_paper(self, filename: str, text: str) -> dict:
        if not text:
            return {}

        # Tokenise text into words and sentences
        words = nltk.word_tokenize(text)
        words_alpha = [w.lower() for w in words if w.isalpha()]
        sentences = nltk.sent_tokenize(text)

        word_count = len(words_alpha)
        sentence_count = len(sentences)

        if word_count == 0 or sentence_count == 0:
            return {}

        # 1. Basic & Structural Metrics
        avg_word_length = np.mean([len(w) for w in words_alpha])
        avg_sentence_length = word_count / sentence_count
        vocab_size = len(set(words_alpha))
        type_token_ratio = vocab_size / word_count # Measures vocabulary richness


        # 2. Part-of-Speech (POS) Tagging (Nouns, Verbs, Adjectives)
        doc = nlp(text[:100000]) # Cap at 100,000 characters for performance speed
        pos_counts = Counter([token.pos_ for token in doc])
        total_tokens = max(len(doc), 1)

        noun_ratio = pos_counts["NOUN"] / total_tokens
        verb_ratio = pos_counts["VERB"] / total_tokens
        adj_ratio = pos_counts["ADJ"] / total_tokens

        # 3. Readability & Complexity Formulas
        flesch_reading_ease = textstat.flesch_reading_ease(text)
        gunning_fog = textstat.gunning_fog(text)
        smog_index = textstat.smog_index(text)

        # 4. Shannon Entropy (Information Density & Randomness)
         # Character Entropy
        char_counts = Counter(text.lower())
        total_chars = sum(char_counts.values())
        char_entropy = -sum((c / total_chars) * math.log2(c / total_chars) for c in char_counts.values())

        # Word Entropy
        word_counts = Counter(words_alpha)
        word_entropy = -sum((w / word_count) * math.log2(w / word_count) for w in word_counts.values())

        # Return feature dictionary aligned with your Master's thesis schema
        return {
            "filename": filename,
            "word_count": word_count,
            "sentence_count": sentence_count,
            "vocabulary_size": vocab_size,
            "type_token_ratio": round(type_token_ratio, 4),
            "avg_word_length": round(avg_word_length, 4),
            "avg_sentence_length": round(avg_sentence_length, 4),
            "noun_ratio": round(noun_ratio, 4),
            "verb_ratio": round(verb_ratio, 4),
            "adjective_ratio": round(adj_ratio, 4),
            "flesch_reading_ease": round(flesch_reading_ease, 2),
            "gunning_fog": round(gunning_fog, 2),
            "smog_index": round(smog_index, 2),
            "char_entropy": round(char_entropy, 4),
            "word_entropy": round(word_entropy, 4),
        }

print("[✓] Cell 3 Complete: StylometricAnalyzer Engine defined!")

[✓] Cell 3 Complete: StylometricAnalyzer Engine defined!


## **Automated Paper Search &amp; PDF Downloader (Europe PMC API)**

*  Automatically search open-access databases and download PDFs straight to Drive using requests.

In [4]:
import requests
import time
import re
from pathlib import Path

def auto_download_papers(search_query: str, max_papers: int = 5):
    """
    Searches Europe PMC for open-access papers matching your keywords and automatically
    downloads full-text PDFs into Google Drive.
    """
    drive_papers_dir = Path("/content/drive/MyDrive/Papers")
    drive_papers_dir.mkdir(parents=True, exist_ok=True)

    # Base URL for Europe PMC REST API
    base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    full_query = f'({search_query}) AND OPEN_ACCESS:y HAS_PDF:y'
    params = {
        'query': full_query,
        'format': 'json',
        'pageSize': max_papers,
        'resultType': 'core'
    }

    # Set a standard Chrome browser User-Agent header so publishers don't block the request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,application/pdf,*/*;q=0.8'
    }

    print(f"[+] Searching Europe PMC for: '{search_query}'...")
    try:
        response = requests.get(base_url, params=params, headers=headers, timeout=15)
        response.raise_for_status()
        data = response.json()
        results = data.get('resultList', {}).get('result', [])
        print(f"[✓] Found {len(results)} open-access paper(s) matching query.\n")

        downloaded_count = 0
        for idx, paper in enumerate(results, start=1):
            title = paper.get('title', f'paper_{idx}').strip('.').replace('/', '_')
            clean_title = "".join([c for c in title if c.isalnum() or c in (' ', '_', '-')])[:60].strip()
            pdf_filename = f"{clean_title}.pdf"
            save_path = drive_papers_dir / pdf_filename

            # Skip paper if it already exists in Google Drive
            if save_path.exists():
                print(f"  [{idx}] Skipping (Already in Drive): {clean_title}.pdf")
                continue

            # Find PDF link (prefer direct PMC open-access link if available)
            pdf_url = None
            pmcid = paper.get('pmcid')
            if pmcid:
                # Direct PMC open access PDF link
                pdf_url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/pdf/"
            else:
                # Fallback to Europe PMC full text URLs
                ft_urls = paper.get('fullTextUrlList', {}).get('fullTextUrl', [])
                for ft in ft_urls:
                    if ft.get('documentStyle') == 'pdf':
                        pdf_url = ft.get('url')
                        break

            if pdf_url:
                try:
                    print(f"  [{idx}] Downloading: {clean_title}.pdf ...")
                    pdf_resp = requests.get(pdf_url, headers=headers, allow_redirects=True, timeout=30)

                    # Verify content is actually a PDF before saving
                    if pdf_resp.status_code == 200 and len(pdf_resp.content) > 5000:
                        with open(save_path, 'wb') as f:
                            f.write(pdf_resp.content)
                        downloaded_count += 1
                        print(f"  [✓] Downloaded successfully!")
                        time.sleep(1.5)  # Pause to respect server limits
                    else:
                        print(f"  [-] Server returned non-PDF or empty file (Status {pdf_resp.status_code})")
                except Exception as download_err:
                    print(f"  [-] Could not download PDF: {download_err}")
            else:
                print(f"  [{idx}] No direct PDF link found for: {clean_title}")

        print(f"\n[✓] Search complete! Successfully downloaded {downloaded_count} NEW PDF(s) to MyDrive/Papers.")
    except Exception as e:
        print(f"[-] Search API error: {e}")

# Cluster 1: Cardiac / Heart Surgery + LLM / RAG
auto_download_papers('("LLM" OR "RAG" OR "Large Language Model") AND ("cardiac surgery" OR "heart surgery")', max_papers=5)

# Cluster 2: General Surgery + AI / LLM (Comparative Group)
auto_download_papers('("LLM" OR "Large Language Model") AND ("surgery") NOT ("cardiac")', max_papers=3)

[+] Searching Europe PMC for: '("LLM" OR "RAG" OR "Large Language Model") AND ("cardiac surgery" OR "heart surgery")'...
[-] Search API error: HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Read timed out. (read timeout=15)
[+] Searching Europe PMC for: '("LLM" OR "Large Language Model") AND ("surgery") NOT ("cardiac")'...
[✓] Found 3 open-access paper(s) matching query.

  [1] Skipping (Already in Drive): American Society of Anesthesiologists Physical Status Classi.pdf
  [2] Skipping (Already in Drive): ChatGPT concordance with guidelines and tumour board treatme.pdf
  [3] Skipping (Already in Drive): Navigating digital surgical advice evaluating the reliabilit.pdf

[✓] Search complete! Successfully downloaded 0 NEW PDF(s) to MyDrive/Papers.


## **Automated Feature Processing & CSV Update**
*  Scan Google Drive for PDFs, analyse new papers, and update features to CSV.

In [5]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import nltk

# Fix: Silently download the newly required NLTK tokenization resource
nltk.download('punkt_tab', quiet=True)

PAPERS_DIR = "/content/drive/MyDrive/Papers"
PROCESSED_TXT_DIR = "/content/drive/MyDrive/Papers/Processed_Text"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Papers/paper_analysis_features.csv"

# 1. Process and clean all PDFs found in your Google Drive folder
processor = PaperTextProcessor(papers_dir=PAPERS_DIR, output_txt_dir=PROCESSED_TXT_DIR)
paper_records = processor.extract_and_clean_all()

if not paper_records:
    print("\n[-] No PDF files found in MyDrive/Papers. Please check your Drive folder path.")
else:
    # 2. Check existing CSV to avoid re-analysing previously processed papers
    output_path = Path(OUTPUT_CSV_PATH)
    existing_filenames = set()
    existing_df = pd.DataFrame()
    if output_path.exists():
        existing_df = pd.read_csv(output_path)
        if 'filename' in existing_df.columns:
            existing_filenames = set(existing_df['filename'].dropna().unique())
            print(f"\n[i] Found existing dataset with {len(existing_filenames)} previously analyzed paper(s).")

    # 3. Filter for brand-new papers only
    new_records = [r for r in paper_records if r['filename'] not in existing_filenames]
    if not new_records:
        print("\n[✓] All papers in your Drive folder are ALREADY analyzed! No new work needed.")
        print(" Current Dataset Overview:")
        display(existing_df.head())
    else:
        print(f"\n[+] Found {len(new_records)} NEW paper(s) to process...")
        analyzer = StylometricAnalyzer()
        new_dataset = []

        # Extract features for new papers only
        for rec in tqdm(new_records, desc="Analyzing New Papers"):
            features = analyzer.analyze_paper(rec["filename"], rec["clean_text"])
            if features:
                new_dataset.append(features)

        new_df = pd.DataFrame(new_dataset)

        # Merge new paper metrics with existing CSV dataset
        if not existing_df.empty:
            final_df = pd.concat([existing_df, new_df], ignore_index=True)
        else:
            final_df = new_df

        # Save updated master CSV back to Google Drive
        final_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
        print(f"\n[✓] Complete! Master feature dataset updated successfully.")
        print(f" Saved to: {OUTPUT_CSV_PATH}")
        print(f" Total papers in dataset: {len(final_df)}")
        print("\nLatest Additions:")
        display(final_df.tail())


[+] Found 23 PDF paper(s) in: /content/drive/MyDrive/Papers


Extracting & Cleaning PDFs: 100%|██████████| 23/23 [00:39<00:00,  1.73s/it]



[i] Found existing dataset with 13 previously analyzed paper(s).

[+] Found 10 NEW paper(s) to process...


Analyzing New Papers: 100%|██████████| 10/10 [00:18<00:00,  1.82s/it]


[✓] Complete! Master feature dataset updated successfully.
 Saved to: /content/drive/MyDrive/Papers/paper_analysis_features.csv
 Total papers in dataset: 23

Latest Additions:


,filename,word_count,sentence_count,vocabulary_size,type_token_ratio,avg_word_length,avg_sentence_length,noun_ratio,verb_ratio,adjective_ratio,flesch_reading_ease,gunning_fog,smog_index,char_entropy,word_entropy
18,16 Development and external validation of a ma...,6693,482,1698,0.2537,5.8126,13.8859,0.1811,0.0502,0.0793,26.98,16.24,14.42,4.7739,9.1749
19,17 Using artificial intelligence to predict po...,8528,568,1947,0.2283,5.5338,15.0141,0.1530,0.0428,0.0591,28.09,17.49,15.69,4.7547,9.3260
20,18 Large Language Models in Surgery Promise Pi...,6208,561,1934,0.3115,5.8284,11.0660,0.1500,0.0639,0.0649,25.87,16.43,14.51,4.6933,9.4925
21,19 A clinical multimodal vision language found...,12105,941,2484,0.2052,5.8775,12.8640,0.2007,0.0625,0.0743,32.45,16.26,14.58,4.8775,9.4822
22,20 Mitigating hallucinations in healthcare AI ...,9282,609,2067,0.2227,6.2615,15.2414,0.2332,0.0791,0.0786,10.80,19.92,17.13,4.6403,9.3259


### **Summary of Code Purpose &amp; Key Findings**

#### **1\. Purpose of the Pipeline**

* **Automated PDF Text Cleaning**: Extracts text from research PDFs, strips out page numbers, headers, and broken line-hyphenations, and standardises text for NLP analysis[1][2].
* **Stylometric &amp; Feature Mining**: Computes quantitative metrics across vocabulary richness (Type-Token Ratio), grammatical structure (POS noun/verb ratios), text complexity (Flesch Reading Ease, Gunning Fog), and information entropy[1][3].
* **Incremental Dataset Updates**: Processes newly added PDFs automatically and appends their feature vectors into `paper_analysis_features.csv` in Google Drive[1][4].

#### **2\. Existing Paper Dataset Results**

* **High Academic Complexity**: Analysed papers (ranging from 6,734 to 19,107 words) display formal post-graduate reading complexity, with Gunning Fog scores reaching up to 23.54 and Flesch Reading Ease dropping as low as -12.36[5][6].
* **Noun-Heavy Writing Style**: Nouns represent a high proportion of tokens (18.4% to 24.7%), while verb usage remains low (5.5% to 7.8%), reflecting heavy passive domain terminology[5][6].
* **Vocabulary Richness &amp; Structural Consistency**: Type-Token Ratios range from 0.1767 to 0.2684, while character entropy (4.57 to 4.81) and word entropy (8.90 to 9.58) remain tightly consistent across published literature[5][6].

#### **3\. New API Search &amp; Download Results (Hybrid Workflow)**

* **Publisher Restricted Access (HTTP 403 Errors)**: Automated searches via public APIs failed to retrieve full-text PDFs because academic journal publishers block automated Python download requests[6].
* **1-Word File Artifacts**: Blocked downloads resulted in placeholder 1-word text files (e.g. `word_count = 1`), generating dummy metric rows in the CSV dataset[6].
* **Practical Hybrid Solution**: Manually curating papers via google ensures full institutional access, while the Python code handles 100% of text cleaning, feature mining, and dataset updates once PDFs are placed into Drive folder

## **NEW RESULTS**
* **Vocabulary Richness vs. Length Trend**: Shorter, focused papers (e.g. Paper 18 at 6,208 words with TTR = **0.3115**; Paper 13 at 4,940 words with TTR = **0.3012**) display significantly higher unique vocabulary diversity than longer comprehensive papers (e.g. Paper 8 at 16,003 words with TTR = **0.1767**).
* **Consistently Advanced Readability**: Across all 20 papers, Gunning Fog scores consistently hover between **15.13 and 23.54** (requiring post-graduate reading comprehension), while Flesch Reading Ease stays low (**10.8 to 36.7**).